# **Setup**

In [1]:
import os, sys

# Repository information
REPO_NAME = "RecSys-Challenge-2025"
REPO_URL  = f"github.com/Lv1g1/{REPO_NAME}.git"

# Detect environment
IS_COLAB = 'content' in os.getcwd()
IS_KAGGLE = 'kaggle' in os.getcwd()
IS_LOCAL = not (IS_COLAB or IS_KAGGLE)

WORKING_DIR = os.getcwd()

if IS_COLAB:
    WORKING_DIR = "/content"

    # Mount Google Drive
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)

    # Get GitHub token via input
    def get_token():
        from getpass import getpass
        return getpass("GitHub Token: ")

elif IS_KAGGLE:
    WORKING_DIR = "/kaggle/working"

    # Get GitHub token from Kaggle secrets
    def get_token():
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret("Token")

# If local environment assume inside the repo
LOCAL_REPO_PATH = "/home/luigi/RecSys" if IS_LOCAL else os.path.join(WORKING_DIR, REPO_NAME)

# Clone the repository if it doesn't exist
if not os.path.exists(LOCAL_REPO_PATH):
    os.chdir(WORKING_DIR)
    token = get_token()

    !git clone https://{token}@{REPO_URL}
else:
    print("Repo already exists — pulling latest changes")
    os.chdir(LOCAL_REPO_PATH)
    !git pull
    os.chdir(WORKING_DIR)

# Add to Python PATH
if LOCAL_REPO_PATH not in sys.path:
    sys.path.append(LOCAL_REPO_PATH)

Repo already exists — pulling latest changes


Already up to date.


In [2]:
if IS_COLAB or False:  # Set to True if you want to recompile Cython files
    os.chdir(LOCAL_REPO_PATH)
    !python run_compile_all_cython.py
    os.chdir(WORKING_DIR)

In [3]:
if IS_COLAB or IS_KAGGLE:
    !pip install optuna

import optuna

In [4]:
import importlib
import numpy as np

from Challenge import paths
importlib.reload(paths)

from Challenge.hyper_tuning import ModelOptimizer

Running on local — storage at: /home/luigi/RecSys
Running on local — storage at: /home/luigi/RecSys


/home/luigi/RecSys/Challenge/hyper_tuning.py:75: ExperimentalWarning: WilcoxonPruner is experimental (supported from v3.6.0). The interface can change in the future.
  def create_study(self, study_name, direction="maximize", load_if_exists=True, pruner=optuna.pruners.WilcoxonPruner()):
/home/luigi/RecSys/Challenge/hyper_tuning.py:107: ExperimentalWarning: WilcoxonPruner is experimental (supported from v3.6.0). The interface can change in the future.
  def create_and_optimize_study(self, study_name, objective_function, n_trials=50, direction="maximize", load_if_exists=True, pruner=optuna.pruners.WilcoxonPruner()):


# **Load Data**

In [5]:
# Load datasets
folds = paths.load_cv_folds(k=5)

In [6]:
def evaluate_recommender(recommender, at, URM_validation):
    cumulative_recall = 0.0
    num_eval = 0
    
    for user_id in range(URM_validation.shape[0]):
        relevant_items = URM_validation.indices[URM_validation.indptr[user_id]:URM_validation.indptr[user_id+1]]
        
        if len(relevant_items)>0:
            num_eval+=1
            
            recommended_items = recommender.recommend(user_id, cutoff=at)
            
            is_relevant = np.isin(recommended_items, relevant_items, assume_unique=True)
            recall_score = np.sum(is_relevant, dtype=np.float32) / relevant_items.shape[0]

            cumulative_recall += recall_score

    return cumulative_recall / num_eval

# **Hyperparameter search**

In [7]:
from Recommenders.GraphBased.P3alphaRecommender import P3alphaRecommender

optimizer = ModelOptimizer("P3alpha")

STUDY_NAME = P3alphaRecommender.RECOMMENDER_NAME

In [8]:
def objective_function(optuna_trial: optuna.trial.Trial) -> float:
    params = {
        "topK": optuna_trial.suggest_int("topK", 10, 1000),
        "alpha": optuna_trial.suggest_float("alpha", 0.1, 2),
        "normalize_similarity": optuna_trial.suggest_categorical("normalize_similarity", [True, False]),
        "implicit": True,
    }
    
    validation_scores = []
    for fold_idx, (URM_train, URM_validation) in enumerate(folds):
        # Train the recommender
        recommender_instance = P3alphaRecommender(URM_train)
        recommender_instance.fit(**params)
        
        # Evaluate
        score = evaluate_recommender(recommender_instance, at=20, URM_validation=URM_validation)
        validation_scores.append(score)
        
        # Show fold result
        print(f"  Fold {fold_idx+1}/{len(folds)} - Score: {score}")

        # Report intermediate result to Optuna
        optuna_trial.report(score, fold_idx)

        # Ask Optuna to prune if performance is poor
        if optuna_trial.should_prune():
            # Return the average score so far instead of raising TrialPruned,
            # which is a common workaround for WilcoxonPruner.
            return np.mean(validation_scores)
        
    # Log folds performance
    optimizer.log_folds(validation_scores, params)

    # Return the mean CV score for the fully completed trial
    return np.mean(validation_scores)

In [9]:
optuna_study = optimizer.create_and_optimize_study(
    study_name=STUDY_NAME,
    objective_function=objective_function,
    n_trials=100
)

[I 2025-11-20 19:29:04,097] A new study created in RDB with name: P3alphaRecommender


  0%|          | 0/100 [00:00<?, ?it/s]

P3alphaRecommender: Similarity column 6969 (100.0%), 2913.21 column/sec. Elapsed time 2.39 sec
  Fold 1/5 - Score: 0.17934240400791168
P3alphaRecommender: Similarity column 6969 (100.0%), 992.05 column/sec. Elapsed time 7.02 sec
  Fold 2/5 - Score: 0.1788986772298813
P3alphaRecommender: Similarity column 6969 (100.0%), 2850.86 column/sec. Elapsed time 2.44 sec
  Fold 3/5 - Score: 0.17988120019435883
P3alphaRecommender: Similarity column 6969 (100.0%), 921.19 column/sec. Elapsed time 7.57 sec
  Fold 4/5 - Score: 0.17866075038909912
P3alphaRecommender: Similarity column 6969 (100.0%), 2774.28 column/sec. Elapsed time 2.51 sec
  Fold 5/5 - Score: 0.17884469032287598
[I 2025-11-20 19:29:51,884] Trial 0 finished with value: 0.17912554740905762 and parameters: {'topK': 197, 'alpha': 0.7462431749828984, 'normalize_similarity': False}. Best is trial 0 with value: 0.17912554740905762.
P3alphaRecommender: Similarity column 6969 (100.0%), 1845.75 column/sec. Elapsed time 3.78 sec
  Fold 1/5 - Sco

In [10]:
optuna.visualization.plot_optimization_history(optuna_study)

In [11]:
optuna.visualization.plot_param_importances(optuna_study)

In [12]:
optuna.visualization.plot_parallel_coordinate(optuna_study)

# **Hyperparameter tuning**

In [13]:
def refined_objective(optuna_trial: optuna.trial.Trial) -> float:
    params = {
        "topK": optuna_trial.suggest_int("topK", 150, 180),
        "alpha": optuna_trial.suggest_float("alpha", 1.56, 1.7),
        "normalize_similarity": True,
        "implicit": True,
    }
    
    validation_scores = []
    for fold_idx, (URM_train, URM_validation) in enumerate(folds):
        # Train the recommender
        recommender_instance = P3alphaRecommender(URM_train)
        recommender_instance.fit(**params)
        
        # Evaluate
        score = evaluate_recommender(recommender_instance, at=20, URM_validation=URM_validation)
        validation_scores.append(score)
        
        # Show fold result
        print(f"  Fold {fold_idx+1}/{len(folds)} - Score: {score}")

        # Report intermediate result to Optuna
        optuna_trial.report(score, fold_idx)

        # Ask Optuna to prune if performance is poor
        if optuna_trial.should_prune():
            # Return the average score so far instead of raising TrialPruned,
            # which is a common workaround for WilcoxonPruner.
            return np.mean(validation_scores)
        
    # Log folds performance
    optimizer.log_folds(validation_scores, params)

    # Return the mean CV score for the fully completed trial
    return np.mean(validation_scores)

In [14]:
optuna_study = optimizer.create_and_optimize_study(
    study_name=STUDY_NAME+"_refined",
    objective_function=refined_objective,
    n_trials=20
)

[I 2025-11-20 20:44:07,854] A new study created in RDB with name: P3alphaRecommender_refined


  0%|          | 0/20 [00:00<?, ?it/s]

P3alphaRecommender: Similarity column 6969 (100.0%), 1095.62 column/sec. Elapsed time 6.36 sec
  Fold 1/5 - Score: 0.2330283373594284
P3alphaRecommender: Similarity column 6969 (100.0%), 2982.78 column/sec. Elapsed time 2.34 sec
  Fold 2/5 - Score: 0.23295488953590393
P3alphaRecommender: Similarity column 6969 (100.0%), 3007.74 column/sec. Elapsed time 2.32 sec
  Fold 3/5 - Score: 0.2344963103532791
P3alphaRecommender: Similarity column 6969 (100.0%), 3223.67 column/sec. Elapsed time 2.16 sec
  Fold 4/5 - Score: 0.23370982706546783
P3alphaRecommender: Similarity column 6969 (100.0%), 3053.94 column/sec. Elapsed time 2.28 sec
  Fold 5/5 - Score: 0.2350713312625885
[I 2025-11-20 20:44:51,058] Trial 0 finished with value: 0.23385211825370789 and parameters: {'topK': 151, 'alpha': 1.6232961706528812}. Best is trial 0 with value: 0.23385211825370789.
P3alphaRecommender: Similarity column 6969 (100.0%), 2848.78 column/sec. Elapsed time 2.45 sec
  Fold 1/5 - Score: 0.23361527919769287
P3alpha

In [15]:
optuna.visualization.plot_optimization_history(optuna_study)

In [16]:
optuna.visualization.plot_param_importances(optuna_study)

In [17]:
optuna.visualization.plot_parallel_coordinate(optuna_study)

## **Best Model**
- Trial 1:
Best Value: 0.2344038039445877
Best Params: {'topK': 167, 'alpha': 1.6313028663201836, 'normalize_similarity': True}

- Trial 2:
Best Value: 0.23443999886512756
Best Params: {'topK': 169, 'alpha': 1.5754208768966982}

Best Value: 0.23443999886512756

Best Params: {'topK': 169, 'alpha': 1.5754208768966982, 'normalize_similarity': True}